# Capstone Project — Exploratory Data Analysis

## Introduction

This project applies the complete NumPy, Pandas, and visualization toolkit to a real-world dataset. You'll follow the OSEMN framework (Obtain, Scrub, Explore, Model, iNterpret), focusing on the first three phases — data loading, cleaning, and exploration — which constitute the majority of a data scientist's work.

**Dataset:** King County House Sales (Seattle area, 2014–2015)  
**Goal:** Understand which features drive home prices.

## Objectives

You will be able to:

* Load and profile a real dataset
* Identify and handle missing values, duplicates, and outliers
* Compute descriptive statistics and correlations
* Create a multi-plot EDA dashboard
* Derive and communicate insights

---

## Phase 1 — Obtain

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:,.2f}'.format)

# The King County dataset is publicly available on Kaggle.
# If you have it locally:
# df = pd.read_csv('data/kc_house_data.csv', parse_dates=['date'])

# For this notebook we generate a realistic simulation so it runs without downloads:
np.random.seed(42)
n = 2000

sqft_living = np.random.normal(2000, 700, n).clip(500, 8000).round(-1).astype(int)
bedrooms    = np.random.choice([1, 2, 3, 3, 3, 4, 4, 5, 6], n)
bathrooms   = (bedrooms * np.random.uniform(0.5, 1.0, n)).round(1)
floors      = np.random.choice([1, 1, 1.5, 2, 2, 2.5, 3], n)
waterfront  = np.random.choice([0, 0, 0, 0, 0, 0, 0, 0, 0, 1], n)
grade       = np.random.randint(4, 13, n)
yr_built    = np.random.randint(1900, 2015, n)
zipcode     = np.random.choice([98001, 98004, 98007, 98033, 98052, 98072, 98112, 98199], n)

price = (
    sqft_living * 150
    + bedrooms * 10000
    + bathrooms * 15000
    + waterfront * 400000
    + grade * 20000
    + np.random.normal(0, 80000, n)
).clip(80000, 3000000).round(-3).astype(int)

# Inject some missing values and an outlier
df = pd.DataFrame({
    'price': price,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'sqft_living': sqft_living,
    'floors': floors,
    'waterfront': waterfront,
    'grade': grade,
    'yr_built': yr_built,
    'zipcode': zipcode,
})

# Introduce missing values
df.loc[np.random.choice(df.index, 50), 'bathrooms'] = np.nan
df.loc[np.random.choice(df.index, 30), 'yr_built'] = np.nan
# Introduce a data entry error
df.loc[5, 'bedrooms'] = 33

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head()

---

## Phase 2 — Scrub (Clean)

In [ ]:
print("=== Initial Data Profile ===")
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum()}")

In [ ]:
print("=== Outlier Detection ===")
print(f"Max bedrooms: {df['bedrooms'].max()} — possible data entry error")
print(df[df['bedrooms'] > 10])

In [ ]:
# Fix bedroom outlier (33 → 3 is likely a typo)
df.loc[df['bedrooms'] > 10, 'bedrooms'] = df.loc[df['bedrooms'] > 10, 'bedrooms'] // 10
print(f"Max bedrooms after fix: {df['bedrooms'].max()}")

# Handle missing values
# bathrooms: impute with median (continuous, skewed)
df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())

# yr_built: impute with median (ordinal, no meaningful mean)
df['yr_built'] = df['yr_built'].fillna(df['yr_built'].median()).astype(int)

print(f"\nMissing after imputation:\n{df.isnull().sum()}")

In [ ]:
# Derived features
df['age'] = 2015 - df['yr_built']
df['price_per_sqft'] = (df['price'] / df['sqft_living']).round(0).astype(int)

print("Derived features added: age, price_per_sqft")
print(df[['price', 'sqft_living', 'age', 'price_per_sqft']].describe())

---

## Phase 3 — Explore

In [ ]:
# === Target variable: price ===
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution of price
sns.histplot(df['price'], kde=True, ax=axes[0])
axes[0].axvline(df['price'].median(), color='red', linestyle='--',
                linewidth=1.5, label=f"Median: ${df['price'].median():,.0f}")
axes[0].axvline(df['price'].mean(), color='orange', linestyle='--',
                linewidth=1.5, label=f"Mean: ${df['price'].mean():,.0f}")
axes[0].set_title('Price Distribution (right-skewed)')
axes[0].set_xlabel('Sale Price ($)')
axes[0].legend()

# Log-transformed price
sns.histplot(np.log1p(df['price']), kde=True, ax=axes[1], color='coral')
axes[1].set_title('Log(Price+1) — Closer to Normal')
axes[1].set_xlabel('log(Price)')

plt.suptitle('Target Variable: Sale Price', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Skewness:  {df['price'].skew():.2f}")
print(f"Kurtosis:  {df['price'].kurtosis():.2f}")
print(f"Right tail: {(df['price'] > 1_000_000).sum()} homes over $1M")

In [ ]:
# === Correlation with price ===
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'floors', 'grade', 'age', 'waterfront']
corr_with_price = df[numeric_features + ['price']].corr()['price'].drop('price').sort_values(ascending=False)

plt.figure(figsize=(8, 4))
colors = ['steelblue' if c > 0 else 'tomato' for c in corr_with_price]
plt.barh(corr_with_price.index[::-1], corr_with_price.values[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlations with Sale Price', fontweight='bold')
plt.xlabel('Pearson r')
plt.tight_layout()
plt.show()

print(corr_with_price)

In [ ]:
# === Price vs top features ===
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# sqft_living vs price — strongest predictor
axes[0, 0].scatter(df['sqft_living'], df['price'], alpha=0.2, s=10, color='steelblue')
z = np.polyfit(df['sqft_living'], df['price'], 1)
p = np.poly1d(z)
x_range = np.linspace(df['sqft_living'].min(), df['sqft_living'].max(), 100)
axes[0, 0].plot(x_range, p(x_range), 'r-', linewidth=2)
axes[0, 0].set_xlabel('Sq Ft Living')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].set_title(f'Sqft Living (r={corr_with_price["sqft_living"]:.2f})')

# Grade vs price — boxplot
sns.boxplot(data=df, x='grade', y='price', ax=axes[0, 1], palette='Blues')
axes[0, 1].set_title('Price by Grade')
axes[0, 1].set_xlabel('Construction Grade')
axes[0, 1].set_ylabel('Price ($)')

# Bedrooms vs price
sns.boxplot(data=df, x='bedrooms', y='price', ax=axes[1, 0])
axes[1, 0].set_title('Price by Bedrooms')
axes[1, 0].set_xlabel('Bedrooms')

# Waterfront premium
wf_labels = {0: 'No Waterfront', 1: 'Waterfront'}
df_wf = df.copy()
df_wf['waterfront_label'] = df['waterfront'].map(wf_labels)
sns.boxplot(data=df_wf, x='waterfront_label', y='price', ax=axes[1, 1],
            palette=['steelblue', 'coral'])
axes[1, 1].set_title('Waterfront Price Premium')
axes[1, 1].set_xlabel('')

wf_median = df.groupby('waterfront')['price'].median()
premium = wf_median[1] / wf_median[0] - 1
axes[1, 1].set_title(f'Waterfront Premium: {premium*100:.0f}% higher median')

plt.suptitle('Key Price Drivers', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === Price by ZIP code ===
zip_summary = df.groupby('zipcode')['price'].agg(['median', 'count']).sort_values('median', ascending=False)
zip_summary.columns = ['Median Price', 'Count']

plt.figure(figsize=(9, 4))
plt.bar(zip_summary.index.astype(str), zip_summary['Median Price'], color='steelblue')
plt.title('Median Sale Price by ZIP Code', fontweight='bold')
plt.ylabel('Median Price ($)')
plt.xlabel('ZIP Code')
plt.tight_layout()
plt.show()

print(zip_summary)

---

## Phase 4 — Key Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS — King County House Sales EDA")
print("=" * 60)

print(f"""
Dataset: {len(df):,} home sales

1. PRICE DISTRIBUTION
   Median price: ${df['price'].median():,.0f}
   Mean price:   ${df['price'].mean():,.0f}  (higher due to right skew)
   Range:        ${df['price'].min():,} — ${df['price'].max():,}
   Recommendation: use log(price) as target for regression models

2. STRONGEST PREDICTORS (by Pearson r)
   - sqft_living (r={corr_with_price['sqft_living']:.2f}): largest driver
   - grade       (r={corr_with_price['grade']:.2f}): construction quality premium
   - bathrooms   (r={corr_with_price['bathrooms']:.2f}): correlated with size

3. WATERFRONT PREMIUM
   Waterfront homes: ${df[df['waterfront']==1]['price'].median():,.0f} median
   Non-waterfront:   ${df[df['waterfront']==0]['price'].median():,.0f} median
   Premium: {(df[df['waterfront']==1]['price'].median() / df[df['waterfront']==0]['price'].median() - 1)*100:.0f}%

4. DATA QUALITY ISSUES FOUND AND FIXED
   - 1 bedroom entry error (33 → 3)
   - 50 missing bathroom values → imputed with median
   - 30 missing yr_built values → imputed with median
""")

---

## Your Turn — Extensions

In [ ]:
# Extension 1: Is there a relationship between house age and price?
# Create a scatter plot of age vs price with a regression line


In [ ]:
# Extension 2: Price per sqft analysis
# Which ZIP code has the highest price-per-sqft?
# Is it the same as the highest absolute price?


In [ ]:
# Extension 3: Create a full correlation heatmap of all numeric features
# Use sns.heatmap with annot=True


In [ ]:
# Extension 4: Segment analysis
# Create 'price_tier' column: 'Budget' (<$300k), 'Mid' ($300-600k), 'Premium' (>$600k)
# Compare average sqft_living and grade across tiers


## Summary

This project applied the full Section 03 toolkit:

| Phase | Tools used |
|-------|------------|
| Obtain | `pd.read_csv()`, `.head()`, `.info()`, `.describe()` |
| Scrub | `.isnull()`, `.fillna()`, boolean indexing for outliers |
| Explore | `.corr()`, `.groupby()`, Matplotlib subplots, Seaborn plots |
| Interpret | Domain reasoning + summary statistics |

The next module covers **Classical Machine Learning** — where you'll take cleaned, explored datasets like this one and build predictive models.